In [1]:
import json
import os
from datasets import load_dataset


ds = load_dataset("PrimeIntellect/verifiable-coding-problems", split="train", trust_remote_code=True)
print(ds[0].keys())

/root/miniconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 144169/144169 [03:13<00:00, 746.47 examples/s] 

dict_keys(['source', 'task_type', 'in_source_id', 'prompt', 'gold_standard_solution', 'verification_info', 'metadata', 'problem_id'])


In [2]:
import ast
from pprint import pprint
dataset = []
for entry in ds:
    gold_standard_solution = entry["gold_standard_solution"]
    if gold_standard_solution is None:
        continue
    if gold_standard_solution.startswith("```python") and gold_standard_solution.endswith("```"):
        tests = entry["verification_info"]        
        if isinstance(tests, str):
            try:
                tests = ast.literal_eval(tests)
            except (ValueError, SyntaxError) as e:
                #Try Json loads instread
                try: 
                    tests = json.loads(entry["verification_info"])
                except (json.JSONDecodeError, SyntaxError, ValueError) as e:
                    print(repr(entry["verification_info"]))
                    print(f"Error in json.loads: {e}")
                    continue
        assert isinstance(tests, dict), "Tests should be a dictionary"
        assert tests['language'] == 'python'
        tests = tests["test_cases"]
        if len(tests) <= 4:
            continue 
        new_entry = {
            "problem": entry["prompt"],
            "solution": gold_standard_solution,
            "tests":tests,
        }

        # Assert tests is a dictionary 
        assert isinstance(tests, list), "Tests should be a dictionary"
        assert "input" in tests[0], "Tests should have an input key"
        assert "output" in tests[0], "Tests should have an output key"

        dataset.append(new_entry)

print(len(dataset))
print(dataset[0])
dataset = dataset
output_dir = os.path.abspath("../../train/code")
output_file = os.path.join(output_dir, "primeintellect.json")
with open(output_file, "w") as f:
    json.dump(dataset, f, indent=4)

23891
{'problem': "Solve the following coding problem using the programming language python:\n\nThere are some websites that are accessible through several different addresses. For example, for a long time Codeforces was accessible with two hostnames codeforces.com and codeforces.ru.\n\nYou are given a list of page addresses being queried. For simplicity we consider all addresses to have the form http://<hostname>[/<path>], where:\n\n  <hostname>\xa0— server name (consists of words and maybe some dots separating them),  /<path>\xa0— optional part, where <path> consists of words separated by slashes. \n\nWe consider two <hostname> to correspond to one website if for each query to the first <hostname> there will be exactly the same query to the second one and vice versa\xa0— for each query to the second <hostname> there will be the same query to the first one. Take a look at the samples for further clarifications.\n\nYour goal is to determine the groups of server names that correspond to

In [ ]:
with  open('../../../../tests/rllm/rewards/primeintellect_test_err.json', "r") as f:
    bad_problems = json.load(f)
len(bad_problems)
from pprint import pprint
pprint(bad_problems[0])


In [ ]:
from rllm.utils import RAG
good_problems = [True] * len(dataset)
rag = RAG(docs=[r['problem'] for r in dataset])


In [7]:
for b in bad_problems:
    results = rag.top_k(b['problem'], k=3)
    
    bad_index = results[0]['idx']
    sim_score = results[0]['score']
    assert sim_score >= 0.99, "Similarity score should be greater than 0.99"
    
    good_problems[bad_index] = False


In [ ]:
# Filter out bad problems
good_dataset = [dataset[i] for i, good in enumerate(good_problems) if good]
print(len(good_dataset))
# Save the good dataset
output_dir = os.path.abspath("../../train/code")
output_file = os.path.join(output_dir, "primeintellect.json")
with open(output_file, "w") as f:
    json.dump(good_dataset, f, indent=4)

